# Notebook 09: DeepLab v3+

**Module:** 07 Segmentation  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Understand atrous/dilated convolution
2. Know ASPP (Atrous Spatial Pyramid Pooling)
3. Compare DeepLab vs UNet for GeoSpatial

## DeepLab v3+ (Chen et al., 2018)

**Atrous convolution:** Same kernel, larger receptive field without downsampling.

**ASPP:** Parallel atrous conv at rates (6, 12, 18) + global average pool → multi-scale context.

**Encoder-Decoder:** DeepLabv3 + UNet-style decoder for sharper boundaries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=(6, 12, 18)):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv2d(in_ch, out_ch, 1),
            *[nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r) for r in rates],
            nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1)),
        ])
        self.project = nn.Conv2d(out_ch * (len(rates) + 2), out_ch, 1)
    def forward(self, x):
        h, w = x.shape[2:]
        feats = [c(x) if i < len(self.convs)-1 else F.interpolate(c(x), (h,w)) for i,c in enumerate(self.convs)]
        return self.project(torch.cat(feats, dim=1))

aspp = ASPP(256, 256)
print(f'ASPP output: {aspp(torch.randn(1,256,32,32)).shape}')

---

## Summary

DeepLab uses atrous conv + ASPP for multi-scale context — good for objects at varying scales.

**Next:** [10_PSPNet.ipynb](10_PSPNet.ipynb)